# Broadcast data across clouds

Copy a 300 GB object from one cloud region to six or seven others as cheaply as
possible. A coding agent rewrites a naive routing function, a fixed grader prices
its routes, and Meta-Evolve keeps the best of ten attempts.

These are the cells of the [Cloud broadcast docs page](https://sentient-xyz.github.io/meta-evolve-docs/applications/cloudcast/). You need Python 3.12+ and
Docker running; the agent also needs the [OpenCode](https://opencode.ai) CLI and
`OPENROUTER_API_KEY`. The saved outputs are from the recorded run.

## Set up

Install the dependencies and download the task (`cloudcast.py`) and the agent (`opencode_agent.py`).

In [ ]:
%pip install -q networkx==3.6.1 "meta-evolve @ git+https://github.com/sentient-xyz/meta-evolve.git@69354f9ebb66df37e5bfc78c26aecff2d1d78fae"

from urllib.request import urlretrieve

BASE = "https://sentient-xyz.github.io/meta-evolve-docs/applications/cloudcast/"
for name in ["cloudcast.py", "opencode_agent.py"]:
    urlretrieve(BASE + name, name)

**What is scored.** A program returns the route each of the object's 10 pieces
takes to each destination. Each link costs its egress price per piece crossing
it, and each region on the routes rents VMs for the transfer. `total_cost` sums
this over five jobs; **lower is better**. A shared link is paid for once, so a
shared tree beats separate copies.

## 1. Load the seed

The ADRS paper's starting program: a separate copy from the source to every destination.

In [ ]:
import meta_evolve as meta
from cloudcast import REFERENCES, SEED_SOURCE, DockerSandbox, evaluate

print(SEED_SOURCE)

def search_algorithm(src, dsts, G, num_partitions):
    """Direct replication: the source sends a full copy to every destination.

    This is the ADRS paper's Cloudcast initial program (appendix C.3); the
    reported 31.1% cost reduction is measured against it.
    """
    topology = BroadCastTopology(src, dsts, num_partitions)
    for dst in dsts:
        edge = G[src][dst]
        for partition in range(num_partitions):
            topology.set_dst_partition_paths(dst, partition, [[src, dst, edge]])
    return topology


## 2. Score a candidate

Candidates run in a locked-down container; the grader prices their routes outside it.
Scoring the seed and the two released baselines checks that Docker works.

In [ ]:
sandbox = DockerSandbox()

def score(source):
    return evaluate(source, sandbox=sandbox)

for name, source in REFERENCES.items():
    print(f"{name:<24} {score(source).metrics['total_cost']:.2f}")

direct_replication_seed  1199.17
released_dijkstra        1045.86
released_mdst            682.97


## 3. Propose with a coding agent

Each trial, the agent edits the program and tests it with `python agent_check.py`, for up to 10 steps.

In [ ]:
from opencode_agent import OpenCodeAgent

agent = OpenCodeAgent("cloudcast_run/agent", score)

## 4. Try ten trials

This makes ten real agent sessions.

In [ ]:
from meta_evolve.policies import TreeSearch

experiment = meta.Experiment(
    task=meta.Task(
        evaluator=score,
        artifact=meta.Text,
        objectives=(meta.Minimize("total_cost"),),
        budget=meta.Budget(trials=10, spend_micros=1_000_000),  # stop at 10 trials or $1
    ),
    seed=meta.Text(SEED_SOURCE),
    proposer=agent,
    search=TreeSearch(max_trials=10),
    context=meta.BestSibling(max_records=40, max_chars=60_000),
    experience=meta.PullAccess(max_operations=1, max_results=40, max_records=60, max_chars=120_000),
    random_seed=20260922,
)
result = meta.run(experiment)

for trial in result.trials():
    cost = trial.metrics.get("total_cost")
    print(f"trial {trial.logical_step}: " + (f"{cost:.2f}" if cost is not None else trial.failure.kind))

trial 0: 1199.17
trial 1: 712.46
trial 2: 711.08
trial 3: 711.49
trial 4: 711.08
trial 5: 830.20
trial 6: 711.08
trial 7: 711.08
trial 8: 674.24
trial 9: 667.81
trial 10: 711.08


## 5. Check it on unseen jobs

In [ ]:
from cloudcast import held_out_jobs

unseen = held_out_jobs()
for name, source in {**REFERENCES, "meta_evolve": result.best().value}.items():
    print(f"{name:<24} {evaluate(source, sandbox=sandbox, jobs=unseen).metrics['total_cost']:.2f}")

direct_replication_seed  1431.36
released_dijkstra        1416.72
released_mdst            840.45
meta_evolve              870.23


Trial 9 was selected: 39.2% cheaper than the seed on the unseen jobs (the ADRS paper
reports 31.1%). See [The recorded run](https://sentient-xyz.github.io/meta-evolve-docs/applications/cloudcast/#the-recorded-run) and
[Try your own routing](https://sentient-xyz.github.io/meta-evolve-docs/applications/cloudcast/#try-your-own-routing).